# Stanza vs Gold UD Analysis

This notebook compares Pipeline v1 outputs from gold UD dev input against Pipeline v1 outputs from Stanza-parsed dev input.

**Gold inputs:** `results/dev_baseline_all.csv`, `results/dev_baseline_meaningful.csv`

**Stanza inputs:** `results/stanza_dev_baseline_all.csv`, `results/stanza_dev_baseline_meaningful.csv`

Analysis only. No verifier, mapper, parser, or correction logic is changed here.

**Purpose:** identify where Stanza parser output changes pipeline decisions, rule firing, dependency labels, or case-marker evidence. This notebook supports later error analysis and future correction design.

## 1. Load CSV Files

In [1]:
import csv
from collections import Counter, defaultdict
from pathlib import Path

RESULTS_DIR = Path("../results")

GOLD_ALL_PATH = RESULTS_DIR / "dev_baseline_all.csv"
GOLD_MEANINGFUL_PATH = RESULTS_DIR / "dev_baseline_meaningful.csv"
STANZA_ALL_PATH = RESULTS_DIR / "stanza_dev_baseline_all.csv"
STANZA_MEANINGFUL_PATH = RESULTS_DIR / "stanza_dev_baseline_meaningful.csv"


def load_csv(filepath):
    with open(filepath, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


gold_all = load_csv(GOLD_ALL_PATH)
gold_meaningful = load_csv(GOLD_MEANINGFUL_PATH)
stanza_all = load_csv(STANZA_ALL_PATH)
stanza_meaningful = load_csv(STANZA_MEANINGFUL_PATH)

print(f"Gold all rows:        {len(gold_all)}")
print(f"Stanza all rows:      {len(stanza_all)}")
print(f"Gold meaningful rows: {len(gold_meaningful)}")
print(f"Stanza meaningful:    {len(stanza_meaningful)}")

Gold all rows:        35217
Stanza all rows:      35217
Gold meaningful rows: 7019
Stanza meaningful:    7009


## 2. Basic Distributions

In [2]:
def print_counter(title, counter, total=None):
    print(title)
    print("=" * len(title))
    for key, count in sorted(counter.items(), key=lambda item: (-item[1], str(item[0]))):
        key_text = str(key) if key else "(blank)"
        if total:
            pct = 100 * count / total
            print(f"  {key_text:<40} {count:>7}  ({pct:>6.2f}%)")
        else:
            print(f"  {key_text:<40} {count:>7}")
    print()


gold_final_counts = Counter(row["final_decision"] for row in gold_all)
stanza_final_counts = Counter(row["final_decision"] for row in stanza_all)
gold_rule_counts = Counter(row["verifier_rule_id"] for row in gold_all if row["verifier_rule_id"])
stanza_rule_counts = Counter(row["verifier_rule_id"] for row in stanza_all if row["verifier_rule_id"])

print_counter("Gold final_decision distribution", gold_final_counts, len(gold_all))
print_counter("Stanza final_decision distribution", stanza_final_counts, len(stanza_all))
print_counter("Gold verifier_rule_id distribution", gold_rule_counts, sum(gold_rule_counts.values()))
print_counter("Stanza verifier_rule_id distribution", stanza_rule_counts, sum(stanza_rule_counts.values()))

Gold final_decision distribution
  no_decision                                28198  ( 80.07%)
  mapping_hypothesis                          4464  ( 12.68%)
  confirmed                                   1715  (  4.87%)
  ambiguous                                    840  (  2.39%)

Stanza final_decision distribution
  no_decision                                28208  ( 80.10%)
  mapping_hypothesis                          4404  ( 12.51%)
  confirmed                                   1741  (  4.94%)
  ambiguous                                    864  (  2.45%)

Gold verifier_rule_id distribution
  R2                                           847  ( 33.15%)
  R1                                           556  ( 21.76%)
  R5                                           499  ( 19.53%)
  R4                                           341  ( 13.35%)
  R3                                           312  ( 12.21%)

Stanza verifier_rule_id distribution
  R2                                           864 

## 3. Build Occurrence-Safe Join Keys

Rows are matched by `sent_id`, `token_form`, and an occurrence index within each `(sent_id, token_form)` group. The occurrence index prevents incorrect joins when the same token form appears multiple times in one sentence.

In [3]:
def add_occurrence_index(rows):
    counts = defaultdict(int)
    indexed_rows = []
    for row in rows:
        base_key = (row["sent_id"], row["token_form"])
        counts[base_key] += 1
        indexed = dict(row)
        indexed["occurrence_index"] = str(counts[base_key])
        indexed["join_key"] = "\t".join((row["sent_id"], row["token_form"], indexed["occurrence_index"]))
        indexed_rows.append(indexed)
    return indexed_rows


gold_indexed = add_occurrence_index(gold_all)
stanza_indexed = add_occurrence_index(stanza_all)

gold_by_key = {row["join_key"]: row for row in gold_indexed}
stanza_by_key = {row["join_key"]: row for row in stanza_indexed}

matched_keys = sorted(set(gold_by_key) & set(stanza_by_key))
unmatched_gold_keys = sorted(set(gold_by_key) - set(stanza_by_key))
unmatched_stanza_keys = sorted(set(stanza_by_key) - set(gold_by_key))

print(f"Matched rows:          {len(matched_keys)}")
print(f"Unmatched gold rows:   {len(unmatched_gold_keys)}")
print(f"Unmatched Stanza rows: {len(unmatched_stanza_keys)}")

Matched rows:          35217
Unmatched gold rows:   0
Unmatched Stanza rows: 0


## 4. Create Matched Comparison Rows

In [4]:
COMPARISON_FIELDS = [
    "sent_id",
    "occurrence_index",
    "sentence_text",
    "token_form",
    "gold_deprel",
    "stanza_deprel",
    "gold_case_marker",
    "stanza_case_marker",
    "gold_verifier_rule_id",
    "stanza_verifier_rule_id",
    "gold_final_decision",
    "stanza_final_decision",
    "gold_final_candidates",
    "stanza_final_candidates",
    "deprel_agree",
    "case_marker_agree",
    "rule_id_agree",
    "final_decision_agree",
]


def bool_text(value):
    return "yes" if value else "no"


matched_rows = []
for key in matched_keys:
    gold = gold_by_key[key]
    stanza = stanza_by_key[key]
    row = {
        "sent_id": gold["sent_id"],
        "occurrence_index": gold["occurrence_index"],
        "sentence_text": gold["sentence_text"],
        "token_form": gold["token_form"],
        "gold_deprel": gold["deprel"],
        "stanza_deprel": stanza["deprel"],
        "gold_case_marker": gold["case_marker"],
        "stanza_case_marker": stanza["case_marker"],
        "gold_verifier_rule_id": gold["verifier_rule_id"],
        "stanza_verifier_rule_id": stanza["verifier_rule_id"],
        "gold_final_decision": gold["final_decision"],
        "stanza_final_decision": stanza["final_decision"],
        "gold_final_candidates": gold["final_candidates"],
        "stanza_final_candidates": stanza["final_candidates"],
    }
    row["deprel_agree"] = bool_text(row["gold_deprel"] == row["stanza_deprel"])
    row["case_marker_agree"] = bool_text(row["gold_case_marker"] == row["stanza_case_marker"])
    row["rule_id_agree"] = bool_text(row["gold_verifier_rule_id"] == row["stanza_verifier_rule_id"])
    row["final_decision_agree"] = bool_text(row["gold_final_decision"] == row["stanza_final_decision"])
    matched_rows.append(row)

print(f"Comparison rows created: {len(matched_rows)}")

Comparison rows created: 35217


## 5. Agreement Summary

In [5]:
def agreement_stats(rows, field):
    agree_count = sum(1 for row in rows if row[field] == "yes")
    pct = 100 * agree_count / len(rows) if rows else 0
    return agree_count, pct


summary_rows = []
for label, field in [
    ("final_decision", "final_decision_agree"),
    ("verifier_rule_id", "rule_id_agree"),
    ("deprel", "deprel_agree"),
    ("case_marker", "case_marker_agree"),
]:
    agree_count, pct = agreement_stats(matched_rows, field)
    summary_rows.append({
        "comparison": label,
        "agree_count": agree_count,
        "matched_rows": len(matched_rows),
        "agreement_percent": pct,
    })

print(f"Matched rows:          {len(matched_rows)}")
print(f"Unmatched gold rows:   {len(unmatched_gold_keys)}")
print(f"Unmatched Stanza rows: {len(unmatched_stanza_keys)}")
print()
for row in summary_rows:
    print(
        f"{row['comparison']:<18} "
        f"{row['agree_count']:>7}/{row['matched_rows']} "
        f"({row['agreement_percent']:.2f}%)"
    )

Matched rows:          35217
Unmatched gold rows:   0
Unmatched Stanza rows: 0

final_decision       34359/35217 (97.56%)
verifier_rule_id     35011/35217 (99.42%)
deprel               33512/35217 (95.16%)
case_marker          35001/35217 (99.39%)


## 6. Disagreement Tables

In [6]:
final_decision_disagreements = [
    row for row in matched_rows if row["final_decision_agree"] == "no"
]
rule_disagreements = [
    row for row in matched_rows if row["rule_id_agree"] == "no"
]
deprel_disagreements = [
    row for row in matched_rows if row["deprel_agree"] == "no"
]
case_marker_disagreements = [
    row for row in matched_rows if row["case_marker_agree"] == "no"
]

print(f"final_decision disagreements: {len(final_decision_disagreements)}")
print(f"rule_id disagreements:        {len(rule_disagreements)}")
print(f"deprel disagreements:         {len(deprel_disagreements)}")
print(f"case_marker disagreements:    {len(case_marker_disagreements)}")
print()

print_counter(
    "final_decision disagreement pairs",
    Counter((row["gold_final_decision"], row["stanza_final_decision"]) for row in final_decision_disagreements),
)
print_counter(
    "rule_id disagreement pairs",
    Counter((row["gold_verifier_rule_id"], row["stanza_verifier_rule_id"]) for row in rule_disagreements),
)
print_counter(
    "deprel disagreement pairs",
    Counter((row["gold_deprel"], row["stanza_deprel"]) for row in deprel_disagreements),
)
print_counter(
    "case_marker disagreement pairs",
    Counter((row["gold_case_marker"], row["stanza_case_marker"]) for row in case_marker_disagreements),
)

final_decision disagreements: 858
rule_id disagreements:        206
deprel disagreements:         1705
case_marker disagreements:    216

final_decision disagreement pairs
  ('mapping_hypothesis', 'no_decision')        350
  ('no_decision', 'mapping_hypothesis')        302
  ('mapping_hypothesis', 'ambiguous')           47
  ('no_decision', 'confirmed')                  44
  ('ambiguous', 'mapping_hypothesis')           33
  ('no_decision', 'ambiguous')                  28
  ('ambiguous', 'no_decision')                  18
  ('confirmed', 'no_decision')                  16
  ('confirmed', 'mapping_hypothesis')           11
  ('mapping_hypothesis', 'confirmed')            9

rule_id disagreement pairs
  ('', 'R4')                                    48
  ('', 'R2')                                    33
  ('R4', '')                                    29
  ('', 'R5')                                    27
  ('R5', '')                                    22
  ('R2', '')                       

## 7. Save Analysis CSVs

In [7]:
def write_csv(filepath, rows, fieldnames=None):
    filepath.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(filepath, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


unmatched_gold = [gold_by_key[key] for key in unmatched_gold_keys]
unmatched_stanza = [stanza_by_key[key] for key in unmatched_stanza_keys]

write_csv(RESULTS_DIR / "stanza_vs_gold_matched.csv", matched_rows, COMPARISON_FIELDS)
write_csv(RESULTS_DIR / "stanza_vs_gold_final_decision_disagreements.csv", final_decision_disagreements, COMPARISON_FIELDS)
write_csv(RESULTS_DIR / "stanza_vs_gold_rule_disagreements.csv", rule_disagreements, COMPARISON_FIELDS)
write_csv(RESULTS_DIR / "stanza_vs_gold_deprel_disagreements.csv", deprel_disagreements, COMPARISON_FIELDS)
write_csv(RESULTS_DIR / "stanza_vs_gold_case_marker_disagreements.csv", case_marker_disagreements, COMPARISON_FIELDS)
write_csv(RESULTS_DIR / "stanza_vs_gold_unmatched_gold.csv", unmatched_gold)
write_csv(RESULTS_DIR / "stanza_vs_gold_unmatched_stanza.csv", unmatched_stanza)

print("Saved analysis outputs:")
print("  results/stanza_vs_gold_matched.csv")
print("  results/stanza_vs_gold_final_decision_disagreements.csv")
print("  results/stanza_vs_gold_rule_disagreements.csv")
print("  results/stanza_vs_gold_deprel_disagreements.csv")
print("  results/stanza_vs_gold_case_marker_disagreements.csv")
print("  results/stanza_vs_gold_unmatched_gold.csv")
print("  results/stanza_vs_gold_unmatched_stanza.csv")

Saved analysis outputs:
  results/stanza_vs_gold_matched.csv
  results/stanza_vs_gold_final_decision_disagreements.csv
  results/stanza_vs_gold_rule_disagreements.csv
  results/stanza_vs_gold_deprel_disagreements.csv
  results/stanza_vs_gold_case_marker_disagreements.csv
  results/stanza_vs_gold_unmatched_gold.csv
  results/stanza_vs_gold_unmatched_stanza.csv


## 8. Top 20 Disagreement Examples

In [8]:
EXAMPLE_COLS = [
    "sent_id",
    "sentence_text",
    "token_form",
    "gold_deprel",
    "stanza_deprel",
    "gold_case_marker",
    "stanza_case_marker",
    "gold_verifier_rule_id",
    "stanza_verifier_rule_id",
    "gold_final_decision",
    "stanza_final_decision",
]


any_disagreements = [
    row for row in matched_rows
    if row["final_decision_agree"] == "no"
    or row["rule_id_agree"] == "no"
    or row["deprel_agree"] == "no"
    or row["case_marker_agree"] == "no"
]

for i, row in enumerate(any_disagreements[:20], start=1):
    print(f"Example {i}")
    print("=" * 70)
    for col in EXAMPLE_COLS:
        print(f"{col}: {row[col]}")
    print()

Example 1
sent_id: dev-s1
sentence_text: रामायण काल में भगवान राम के पुत्र कुश की राजधानी कुशावती को 483 ईसा पूर्व बुद्ध ने अपने अंतिम विश्राम के लिए चुना ।
token_form: 483
gold_deprel: compound
stanza_deprel: nummod
gold_case_marker: 
stanza_case_marker: 
gold_verifier_rule_id: 
stanza_verifier_rule_id: 
gold_final_decision: no_decision
stanza_final_decision: no_decision

Example 2
sent_id: dev-s1
sentence_text: रामायण काल में भगवान राम के पुत्र कुश की राजधानी कुशावती को 483 ईसा पूर्व बुद्ध ने अपने अंतिम विश्राम के लिए चुना ।
token_form: ईसा
gold_deprel: compound
stanza_deprel: obl
gold_case_marker: 
stanza_case_marker: पूर्व
gold_verifier_rule_id: 
stanza_verifier_rule_id: 
gold_final_decision: no_decision
stanza_final_decision: mapping_hypothesis

Example 3
sent_id: dev-s1
sentence_text: रामायण काल में भगवान राम के पुत्र कुश की राजधानी कुशावती को 483 ईसा पूर्व बुद्ध ने अपने अंतिम विश्राम के लिए चुना ।
token_form: पूर्व
gold_deprel: obl
stanza_deprel: case
gold_case_marker: 
stanza_c

## 9. Notes for Later Error Analysis

- A `deprel` disagreement can change mapper status and rule eligibility.
- A `case_marker` disagreement can directly change verifier rule firing.
- A `verifier_rule_id` disagreement is especially important because it shows rule activation changed between gold UD and Stanza UD.
- A `final_decision` disagreement is the downstream effect of parser differences on the symbolic layer.
- This notebook does not implement correction. It only identifies where a future correction layer may be useful.